# Preprocessing and Feature Engineering
---
## Tasks done in this notebook:
- Feature Creation, Scaling and Encoding
- Class Imbalance Handling 
- Train/Val/Test sets preparation

**Note: All statistics (mean, percentiles, encoders) must be fit on Train set only in order to prevent leakage**

*This notebook working is dependent on the EDA findings*

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import logging

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

RANDOM_STATE = 42 

### Initialization:
- Data Loading 
- Integrity Check 
- Initial Inspection of Data

In [13]:
credit_data_df = pd.read_csv('../data/processed/credit_card_fraud_10k_cleaned.csv')
print("Dataset Shape:", credit_data_df.shape)
print("\nDataset Info:")
print(credit_data_df.info())
print("\nDataset Description:")
print(credit_data_df.describe())

print("\nDuplicate Rows:", credit_data_df.duplicated().sum())
print("\nMissing Values in Each Column:")
print(credit_data_df.isnull().sum())

continuous_cols = ['amount', 'transaction_hour', 'device_trust_score', 'velocity_last_24h', 'cardholder_age']
binary_cols = ['foreign_transaction', 'location_mismatch']
categorical_cols = ['merchant_category']
target_col = ['is_fraud']

target_distribution = credit_data_df[target_col].value_counts(normalize=True)
print("\nTarget Variable Distribution:")
print(target_distribution)

Dataset Shape: (10000, 10)

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       10000 non-null  int64  
 1   amount               10000 non-null  float64
 2   transaction_hour     10000 non-null  int64  
 3   merchant_category    10000 non-null  str    
 4   foreign_transaction  10000 non-null  int64  
 5   location_mismatch    10000 non-null  int64  
 6   device_trust_score   10000 non-null  float64
 7   velocity_last_24h    10000 non-null  float64
 8   cardholder_age       10000 non-null  int64  
 9   is_fraud             10000 non-null  int64  
dtypes: float64(3), int64(6), str(1)
memory usage: 781.4 KB
None

Dataset Description:
       transaction_id        amount  transaction_hour  foreign_transaction  \
count     10000.00000  10000.000000      10000.000000         10000.000000   
mean       5000.50

### Stratified Train/Val/Test Split (Pre-Engineering)

- Splitting the dataset to train with 70% of data, val with 15% of data and test with 15% of the data
- Why split first? Prevents leakage when computing percentiles, merchant baselines, or encoders

In [18]:
train_size = 0.7 
val_size = 0.15
test_size = 0.15

X = credit_data_df.drop(columns=target_col)
Y = credit_data_df[target_col]

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, 
    Y, 
    test_size=(1 - train_size), 
    random_state=RANDOM_STATE, 
    stratify=Y
)   

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=(test_size / (test_size + val_size)),
    random_state=RANDOM_STATE,
    stratify=Y_temp
)

print("\nTraining Set Shape:", X_train.shape)
print("Validation Set Shape:", X_val.shape)
print("Test Set Shape:", X_test.shape)

split_data = {
    'X_train': X_train,
    'Y_train': Y_train,
    'X_val': X_val,
    'Y_val': Y_val,
    'X_test': X_test,
    'Y_test': Y_test
}   

for name, data in split_data.items():
    print(f"{name} - Shape: {data.shape}, Class Distribution:\n{data.value_counts(normalize=True)}\n")

assert X_train.shape[0] == Y_train.shape[0], "Mismatch in training set sizes"
assert X_val.shape[0] == Y_val.shape[0], "Mismatch in validation set sizes"
assert X_test.shape[0] == Y_test.shape[0], "Mismatch in test set sizes"
assert len(X_train) + len(X_val) + len(X_test) == len(credit_data_df), "Total samples in splits do not match original dataset"
assert len(Y_train) + len(Y_val) + len(Y_test) == len(credit_data_df), "Total samples in target splits do not match original dataset"


Training Set Shape: (6999, 9)
Validation Set Shape: (1500, 9)
Test Set Shape: (1501, 9)
X_train - Shape: (6999, 9), Class Distribution:
transaction_id  amount  transaction_hour  merchant_category  foreign_transaction  location_mismatch  device_trust_score  velocity_last_24h  cardholder_age
4344            315.12  17                Food               0                    0                  78.0                1.0                46                0.000143
2144            67.93   6                 Electronics        0                    0                  45.0                1.0                67                0.000143
9845            22.07   13                Clothing           0                    0                  33.0                4.0                44                0.000143
2349            162.10  6                 Food               0                    0                  78.0                4.0                38                0.000143
2091            268.99  1               

## Feature Engineering (Train-Derived Statistics)
---
### X_train is the only set that will be used while doing the feature engineering

Engineered features and their justifications:
- `log_amount`: handles right skew of amount.
- `is_night_transaction`: 12 AM to 5 AM found to be the burst window. (boolean flag)
- `velocity_per_hour`: identify sudden spikes in activity relative to the time of day.
- `is_high_velocity_low_trust`: fraud found to be more often at high velocity and low trust. (boolean flag)
- `high_risk_abroad`: fraud is significantly higher in foreign markets regardless of price.
- `relative_amount`: show if a purchase is "out of character" for that specific category.
- `trust_score_binned`: Low/Medium/High (Ordinal).

How to calculate?
- `log_amount`: log transformation for the amount.
- `is_night_transaction`: if between 12 AM to 5 AM set with 1, else set with 0.
- `velocity_per_hour`: `velocity_last_24h`/(`transaction_hour` + 1).
- `high_risk_abroad`: `foreign_transaction` * `location_mismatch`.
- `is_high_velocity_low_trust`: if (velocity_last_24h > threshold) & (device_trust_score < threshold) then set with 1, else set with 0. Threshold will be a percentile from the velocity so that it becomes self adjusting.
- `relative_amount`: `amount` / `average_merchant_category_amount`. 
- `average_merchant_category_amount`: mean transaction amount for each specific merchant category.
- `trust_score_binned`: Ordinal Encoding
